# Silver Layer — Cleaning & Transformation

Silver Layer adalah tahap **membersihkan dan menstandardisasi** data dari Bronze.
Output Silver harus bisa dipercaya — tidak ada lagi nilai aneh, nama kolom
tidak konsisten, atau skala yang tidak seimbang.

**Kenapa mulai pakai Spark di sini (bukan di Bronze)?**  
Bronze hanya membaca Excel — Spark tidak bisa baca Excel secara native,
jadi Pandas lebih praktis di sana. Tapi Silver melakukan operasi berat:
filter, transformasi, dan agregasi di 66.000+ baris secara berulang.
Di sinilah Spark mulai memberikan keuntungan dibanding Pandas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time, os

# local[*] → pakai semua core CPU yang tersedia di laptop
# Ini yang membuat Spark lebih cepat dari Pandas untuk operasi berat
spark = SparkSession.builder \
    .appName("GHGRP_Silver") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

BRONZE_PATH = "/content/drive/MyDrive/ABD/data/bronze/ghgrp_bronze.parquet"
SILVER_DIR  = "/content/drive/MyDrive/ABD/data/silver"
os.makedirs(SILVER_DIR, exist_ok=True)

print("SparkSession siap")
print(f"Cores yang digunakan: {spark.sparkContext.defaultParallelism}")

Mounted at /content/drive
SparkSession siap
Cores yang digunakan: 2


## Step 1 — Rename kolom

**Kenapa perlu?**  
Nama kolom asli EPA punya spasi trailing (` `) di akhir dan kata-kata
yang panjang. Ini menyulitkan penulisan kode dan rawan typo.
Di Silver kita standardisasi semua nama kolom ke format `snake_case`
yang konsisten — standar yang umum dipakai di pipeline data.

In [ ]:
start_time = time.time()
df = spark.read.parquet(BRONZE_PATH)
print(f"Bronze dibaca: {df.count():,} baris")

rename_map = {
    'Facility Id'                        : 'facility_id',
    'Facility Name'                      : 'facility_name',
    'City'                               : 'city',
    'State'                              : 'state',
    'Primary NAICS Code'                 : 'naics_code',
    'Industry Type (sectors)'            : 'sector',
    'Total reported direct emissions'    : 'total_emissions',
    'CO2 emissions (non-biogenic) '      : 'co2_emissions',
    'Methane (CH4) emissions '           : 'ch4_emissions',
    'Nitrous Oxide (N2O) emissions '     : 'n2o_emissions',
}

for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

print("Rename kolom selesai")

Bronze dibaca: 66,881 baris
Rename kolom selesai


## Step 2 — Hapus duplikat

**Kenapa perlu?**  
Satu fasilitas seharusnya hanya punya satu baris per tahun.
Kalau ada duplikat `(facility_id, year)`, berarti ada entri
yang terbaca dua kali — ini akan menggandakan nilai emisi
dan merusak hasil clustering maupun tren.

**Kenapa deduplikasi berdasarkan `facility_id` + `year`, bukan semua kolom?**  
Karena dua baris bisa punya `facility_id` dan `year` sama
tapi nilai emisinya sedikit berbeda akibat revisi laporan —
ini tetap duplikat yang harus dibuang.

In [ ]:
before = df.count()
df = df.dropDuplicates(['facility_id', 'year'])
after = df.count()
print(f"Duplikat dihapus : {before - after} baris")
print(f"Sisa             : {after:,} baris")

Duplikat dihapus : 0 baris
Sisa             : 66,881 baris


## Step 3 — Filter nilai tidak valid

**Kenapa drop baris dengan `total_emissions` null, bukan diimputasi?**  
`total_emissions` adalah kolom target utama analisis — tidak bisa dikira-kira.
Kalau nilai ini kosong, kita tidak tahu fasilitas itu mengemisikan berapa,
sehingga baris itu tidak bisa berkontribusi pada analisis apapun.
Berbeda dengan `co2_emissions` atau `ch4_emissions` yang bisa diimputasi
karena bukan kolom utama.

**Kenapa filter emisi > 0?**  
Emisi tidak bisa bernilai nol atau negatif secara fisik — fasilitas
yang melaporkan ke GHGRP pasti menghasilkan emisi minimal 25.000 ton CO2e
(itu ambang batas pelaporan EPA). Nilai ≤ 0 adalah anomali data.

In [ ]:
before = df.count()
df = df.filter(F.col('total_emissions').isNotNull())
df = df.filter(F.col('total_emissions') > 0)
after = df.count()
print(f"Baris tidak valid dihapus : {before - after}")
print(f"Sisa                      : {after:,} baris")

Baris tidak valid dihapus : 313
Sisa                      : 66,568 baris


## Step 4 — Standardisasi kolom sektor

**Kenapa perlu?**  
Beberapa fasilitas melaporkan lebih dari satu sektor, contoh:
`"Chemicals,Petroleum Product Suppliers,Refineries"`  
Ini akan terbaca sebagai satu kategori tersendiri saat clustering,
padahal seharusnya masuk ke sektor utamanya.

**Kenapa ambil sektor pertama?**  
Sektor pertama dalam daftar adalah sektor primer fasilitas tersebut
berdasarkan konvensi pelaporan EPA. Ini pendekatan yang paling
konservatif dan dapat dipertanggungjawabkan.

In [ ]:
# Ambil sektor pertama dari kombinasi, trim spasi
df = df.withColumn('sector_clean',
    F.trim(F.split(F.col('sector'), ',')[0])
)

print("Distribusi sektor setelah standardisasi:")
df.groupBy('sector_clean').count() \
  .orderBy(F.desc('count')) \
  .show(10, truncate=40)

Distribusi sektor setelah standardisasi:
+----------------------------------------+-----+
|                            sector_clean|count|
+----------------------------------------+-----+
|                            Power Plants|13566|
|                                   Other|12574|
|       Petroleum and Natural Gas Systems|12406|
|                                   Waste|12288|
|                               Chemicals| 4590|
|                                Minerals| 3766|
|                                  Metals| 2994|
|                          Pulp and Paper| 2203|
|Natural Gas and Natural Gas Liquids S...| 1018|
|             Petroleum Product Suppliers|  750|
+----------------------------------------+-----+
only showing top 10 rows


## Step 5 — Log transformation pada kolom emisi

**Kenapa ini keputusan paling penting di Silver Layer?**

Dari eksplorasi di Bronze, kita tahu distribusi emisi sangat *skewed*:
ada fasilitas yang emisinya 25.000 ton, ada yang 100 juta ton.
Perbedaannya ribuan kali lipat.

Kalau langsung di-clustering tanpa transformasi, K-Means akan
"terpesona" oleh outlier besar dan hampir semua fasilitas
akan masuk cluster "rendah" — clustering tidak bermakna.

**Kenapa log, bukan normalisasi biasa (min-max atau z-score)?**  
Log transformation mengompres rentang nilai yang sangat jauh
menjadi skala yang lebih proporsional. Contoh:
- 25.000 ton → log = 10.1
- 250.000 ton → log = 12.4  
- 25.000.000 ton → log = 17.0

Perbedaan yang tadinya ribuan kali lipat jadi hanya beberapa unit —
K-Means bisa bekerja dengan baik.

Min-max scaling tidak membantu karena hanya menggeser rentang,
bukan mengubah distribusinya.

In [ ]:
# log(x + 1) → +1 untuk menghindari log(0) kalau ada nilai 0
df = df.withColumn('log_emissions', F.log(F.col('total_emissions') + 1))

# Bandingkan statistik sebelum dan sesudah
print("Statistik total_emissions (asli):")
df.select(
    F.mean('total_emissions').alias('mean'),
    F.expr('percentile_approx(total_emissions, 0.5)').alias('median'),
    F.max('total_emissions').alias('max')
).show()

print("Statistik log_emissions (setelah transformasi):")
df.select(
    F.mean('log_emissions').alias('mean_log'),
    F.expr('percentile_approx(log_emissions, 0.5)').alias('median_log'),
    F.max('log_emissions').alias('max_log')
).show()

print("→ Setelah log transform, mean dan median jauh lebih dekat.")
print("  Ini berarti distribusi sudah lebih simetris dan siap untuk clustering.")

Statistik total_emissions (asli):
+-----------------+---------+--------------+
|             mean|   median|           max|
+-----------------+---------+--------------+
|402333.8767622027|64420.608|2.1775439586E7|
+-----------------+---------+--------------+

Statistik log_emissions (setelah transformasi):
+------------------+------------------+------------------+
|          mean_log|        median_log|           max_log|
+------------------+------------------+------------------+
|11.332236236995081|11.073204383765841|16.896293314068753|
+------------------+------------------+------------------+

→ Setelah log transform, mean dan median jauh lebih dekat.
  Ini berarti distribusi sudah lebih simetris dan siap untuk clustering.


## Step 6 — Imputasi gas CO2, CH4, N2O

**Kenapa diimputasi dengan median, bukan mean atau 0?**

- **Mean**: sensitif terhadap outlier — ada beberapa fasilitas dengan
  emisi CH4 sangat besar yang akan menarik mean jauh ke atas.
  Mengisi dengan mean berarti kita mengisi dengan nilai yang
  tidak representatif.

- **0**: tidak masuk akal secara fisik — fasilitas yang
  tidak melaporkan CH4 bukan berarti emisi CH4-nya nol.
  Bisa jadi mereka tidak diwajibkan melaporkan gas itu.

- **Median**: nilai tengah yang tidak terpengaruh outlier.
  Lebih representatif untuk distribusi yang skewed seperti ini.

**Catatan:** Kolom ini hanya digunakan sebagai fitur tambahan
di Gold Layer — bukan kolom target utama — sehingga imputasi
median sudah cukup memadai.

In [ ]:
for gas_col in ['co2_emissions', 'ch4_emissions', 'n2o_emissions']:
    if gas_col in df.columns:
        # Hitung median dari nilai yang ada
        median_val = df.filter(F.col(gas_col).isNotNull()) \
                       .approxQuantile(gas_col, [0.5], 0.01)[0]

        # Isi nilai null dengan median
        null_count = df.filter(F.col(gas_col).isNull()).count()
        df = df.withColumn(gas_col,
            F.when(F.col(gas_col).isNull(), median_val)
             .otherwise(F.col(gas_col))
        )
        print(f"{gas_col}: {null_count:,} null diisi dengan median = {median_val:,.0f}")

# Verifikasi tidak ada null lagi
print("\nVerifikasi missing values setelah imputasi:")
for c in ['total_emissions','co2_emissions','ch4_emissions','n2o_emissions']:
    n = df.filter(F.col(c).isNull()).count()
    print(f"  {c}: {n} null")

co2_emissions: 7,034 null diisi dengan median = 53,523
ch4_emissions: 838 null diisi dengan median = 599
n2o_emissions: 10,702 null diisi dengan median = 38

Verifikasi missing values setelah imputasi:
  total_emissions: 0 null
  co2_emissions: 0 null
  ch4_emissions: 0 null
  n2o_emissions: 0 null


## Step 7 — Simpan Silver Layer

**Kenapa partisi per tahun?**  
Dengan `partitionBy('year')`, Spark menyimpan data dalam
subfolder terpisah per tahun. Ketika Gold Layer nanti
hanya butuh data 2020–2023, Spark cukup baca folder
tahun itu saja — tidak perlu scan seluruh dataset.
Ini yang disebut *partition pruning* dan bisa menghemat
waktu baca secara signifikan.

In [ ]:
silver_path = os.path.join(SILVER_DIR, 'ghgrp_silver.parquet')

df.write \
  .mode('overwrite') \
  .partitionBy('year') \
  .parquet(silver_path)

silver_time = time.time() - start_time

# Hitung ukuran
total_size = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk(silver_path)
    for f in files
) / (1024 * 1024)

print(f"Silver tersimpan  : {silver_path}")
print(f"Total baris       : {df.count():,}")
print(f"Ukuran Parquet    : {total_size:.2f} MB")
print(f"Total waktu Spark : {silver_time:.1f} detik")
print()
print("Kolom yang tersedia di Silver:")
for c in df.columns:
    print(f"  {c}")

spark.stop()

Silver tersimpan  : /content/drive/MyDrive/ABD/data/silver/ghgrp_silver.parquet
Total baris       : 66,568
Ukuran Parquet    : 4.11 MB
Total waktu Spark : 45.9 detik

Kolom yang tersedia di Silver:
  facility_id
  facility_name
  city
  state
  naics_code
  sector
  total_emissions
  co2_emissions
  ch4_emissions
  n2o_emissions
  year
  sector_clean
  log_emissions


## Ringkasan keputusan Silver Layer

| Keputusan | Alasan |
|---|---|
| Pakai Spark (bukan Pandas) | Operasi berat di 66k+ baris, bisa manfaatkan multi-core |
| Deduplikasi per `facility_id + year` | Cegah penghitungan emisi ganda |
| Drop null `total_emissions` | Kolom target — tidak bisa diimputasi |
| Ambil sektor pertama dari kombinasi | Sektor primer fasilitas menurut konvensi EPA |
| Log transformation | Normalisasi distribusi skewed untuk clustering |
| Imputasi median untuk CO2/CH4/N2O | Robust terhadap outlier, lebih representatif dari mean |
| `partitionBy('year')` | Efisiensi baca di Gold Layer (partition pruning) |

**Langkah selanjutnya:** buka `gold_layer.ipynb`